# 02 · HMRC VAT check API (Sandbox)

Research log: [Step 2](../docs/research_log.md#step-2--hmrc-verificatorul-web-și-api-ul-sandbox) (in Romanian). Data and checks as of October 2026.

**Question.** Can the VAT check be automated, and what does HMRC return for a number?

**Method.** HMRC Developer Hub Sandbox application; OAuth client-credentials token; "Check a UK VAT number 2.0" on the 40 Sandbox test numbers (`vrn.csv` from HMRC's GitHub repository) and on two invalid inputs.

**Result (from the research log).**
- 40/40 test numbers return 200 and the same number (including 12-digit numbers and leading zeros).
- `123456789` → 404 NOT_FOUND; `12345` → 400 INVALID_REQUEST.
- The response has name, address and processing date, but no CH number, so attribution must use name and postcode. Production access needs a registered organisation, so real numbers were checked by hand on the web page.

**Notes for re-running.**
- Needs `HMRC_CLIENT_ID` and `HMRC_CLIENT_SECRET` for a Sandbox application in a `.env` file (never committed).

*Cells are kept as they were run, outputs included; only file paths were updated to the new `data/` layout and session timestamps in outputs were cut to the month.*

In [3]:
import os
from dotenv import load_dotenv

found = load_dotenv()
print("Am găsit .env:", found)

client_id = os.getenv("HMRC_CLIENT_ID")
client_secret = os.getenv("HMRC_CLIENT_SECRET")

print("Client ID încărcat:", client_id is not None)
print("Secret încărcat:", client_secret is not None)
print("Lungime secret:", len(client_secret) if client_secret else 0)

Am găsit .env: True
Client ID încărcat: True
Secret încărcat: True
Lungime secret: 36


In [4]:
import requests

TOKEN_URL = "https://test-api.service.hmrc.gov.uk/oauth/token"

response = requests.post(
    TOKEN_URL,
    data={
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
        "scope": "read:vat",
    },
)

print("Status:", response.status_code)
token_data = response.json()
print("Chei primite:", list(token_data.keys()))
print("Expiră în (secunde):", token_data.get("expires_in"))

Status: 200
Chei primite: ['access_token', 'scope', 'expires_in', 'token_type']
Expiră în (secunde): 14400


In [5]:
VAT_URL = "https://test-api.service.hmrc.gov.uk/organisations/vat/check-vat-number/lookup/{vrn}"

access_token = token_data["access_token"]
vrn = "553557881"

response = requests.get(
    VAT_URL.format(vrn=vrn),
    headers={
        "Accept": "application/vnd.hmrc.2.0+json",
        "Authorization": f"Bearer {access_token}",
    },
)

print("Status:", response.status_code)
print(response.json())

Status: 200
{'target': {'name': 'Credite Sberger Donal Inc.', 'vatNumber': '553557881', 'address': {'line1': '131B Barton Hamlet', 'postcode': 'SW97 5CK', 'countryCode': 'GB'}}, 'processingDate': '2026-10'}


In [6]:
import json
print(json.dumps(response.json(), indent=2))

{
  "target": {
    "name": "Credite Sberger Donal Inc.",
    "vatNumber": "553557881",
    "address": {
      "line1": "131B Barton Hamlet",
      "postcode": "SW97 5CK",
      "countryCode": "GB"
    }
  },
  "processingDate": "2026-10"
}


In [7]:
RAW_URL = "https://raw.githubusercontent.com/hmrc/vat-registered-companies-api/refs/heads/main/public/api/conf/2.0/test-data/vrn.csv"

r = requests.get(RAW_URL)
print("Status:", r.status_code)

vrns = [v.strip() for v in r.text.split(",")]

print("Câte numere:", len(vrns))
print("Primele 5:", vrns[:5])
print("Lungimi:", sorted(set(len(v) for v in vrns)))

Status: 200
Câte numere: 40
Primele 5: ['553557881', '146295999727', '948561936944', '166804280212', '711954454070']
Lungimi: [9, 12]


In [8]:
import time
import pandas as pd

def check_vat(vrn):
    return requests.get(
        VAT_URL.format(vrn=vrn),
        headers={
            "Accept": "application/vnd.hmrc.2.0+json",
            "Authorization": f"Bearer {access_token}",
        },
    )

results = []

for vrn in vrns:
    r = check_vat(vrn)
    row = {"vrn_trimis": vrn, "status": r.status_code}
    if r.status_code == 200:
        target = r.json()["target"]
        row["vrn_primit"] = target["vatNumber"]
        row["name"] = target["name"]
        row["postcode"] = target["address"].get("postcode")
    results.append(row)
    time.sleep(0.5)

df = pd.DataFrame(results)
print(df["status"].value_counts())
df

status
200    40
Name: count, dtype: int64


,vrn_trimis,status,vrn_primit,name,postcode
0,553557881,200,553557881,Credite Sberger Donal Inc.,SW97 5CK
1,146295999727,200,146295999727,Chinan,S54 7PO
2,948561936944,200,948561936944,Alpharmart,SO36 9DZ
3,166804280212,200,166804280212,Brid Compo Jard Ltd.,OX17 0VW
4,711954454070,200,711954454070,Compal Holdings Shaa,BB05 5PR
5,355699311774,200,355699311774,Holdin Group,KY41 2PV
6,770555752641,200,770555752641,PCE CIT,KW05 7UG
7,436189915,200,436189915,Phillia Communics Cor,HA60 4RO
8,011591667,200,011591667,Lufthay,NG01 0ZM
9,961925638,200,961925638,SABIC Fuel,CT84 7VH


In [9]:
print("Toate identice:", (df["vrn_trimis"] == df["vrn_primit"]).all())

Toate identice: True


In [10]:
teste = {
    "format corect, inexistent": "123456789",
    "format greșit": "12345",
}

for descriere, vrn in teste.items():
    r = check_vat(vrn)
    print(descriere, "→", vrn)
    print("  Status:", r.status_code)
    print("  Răspuns:", r.text)
    print()

format corect, inexistent → 123456789
  Status: 404
  Răspuns: {"code":"NOT_FOUND","message":"targetVrn does not match a registered company"}

format greșit → 12345
  Status: 400
  Răspuns: {"code":"INVALID_REQUEST","message":"Invalid targetVrn - Vrn parameters should be 9 or 12 digits"}

